# 🥈 Notebook 02 — Silver Layer: Build Dimension Tables

**Goal:** Clean Bronze data and build 5 dimension tables — the backbone of the star schema.

> **Run time:** ~5 min

```
Bronze Tables
  └──► dim_customer   — Who     (500 rows)
  └──► dim_account    — What    (500 rows)
  └──► dim_product    — Product (10  rows)
  └──► dim_branch     — Where   (10  rows)
  └──► dim_date       — When    (auto-generated, 2020-2025)
```

In [ ]:
from pyspark.sql import functions as F

print('Building Silver dimension tables...')

## dim_customer

In [ ]:
dim_customer = spark.table('bronze_customers') \
    .drop('_ingested_at','_source_file') \
    .dropDuplicates(['CustomerID']) \
    .withColumn('FullName', F.concat_ws(' ', 'FirstName', 'LastName')) \
    .withColumn('Age', F.floor(F.months_between(F.current_date(), F.to_date('DateOfBirth','yyyy-MM-dd')) / 12)) \
    .withColumn('AgeGroup',
        F.when(F.col('Age') < 30, 'Under 30')
         .when(F.col('Age') < 45, '30-44')
         .when(F.col('Age') < 60, '45-59')
         .otherwise('60+')) \
    .withColumn('CreditScoreTier',
        F.when(F.col('CreditScore') >= 750, 'Excellent')
         .when(F.col('CreditScore') >= 670, 'Good')
         .when(F.col('CreditScore') >= 580, 'Fair')
         .otherwise('Poor')) \
    .withColumn('JoinYear', F.year(F.to_date('JoinDate','yyyy-MM-dd'))) \
    .withColumn('_updated_at', F.current_timestamp())

dim_customer.write.format('delta').mode('overwrite').saveAsTable('dim_customer')
print(f'dim_customer: {dim_customer.count()} rows')
dim_customer.select('CustomerID','FullName','CreditScoreTier','AgeGroup','CustomerSegment').show(5)

## dim_account

In [ ]:
dim_account = spark.table('bronze_accounts') \
    .drop('_ingested_at','_source_file') \
    .dropDuplicates(['AccountID']) \
    .withColumn('OpenYear', F.year(F.to_date('OpenDate','yyyy-MM-dd'))) \
    .withColumn('BalanceTier',
        F.when(F.col('Balance') >= 100000, 'High Value')
         .when(F.col('Balance') >= 25000,  'Mid Value')
         .when(F.col('Balance') >= 5000,   'Standard')
         .otherwise('Low Balance')) \
    .withColumn('_updated_at', F.current_timestamp())

dim_account.write.format('delta').mode('overwrite').saveAsTable('dim_account')
print(f'dim_account: {dim_account.count()} rows')
dim_account.show(5)

## dim_product

In [ ]:
dim_product = spark.table('bronze_products') \
    .drop('_ingested_at','_source_file') \
    .dropDuplicates(['ProductID']) \
    .withColumn('RateSpread', F.round(F.col('MaxInterestRate') - F.col('MinInterestRate'), 2)) \
    .withColumn('_updated_at', F.current_timestamp())

dim_product.write.format('delta').mode('overwrite').saveAsTable('dim_product')
print(f'dim_product: {dim_product.count()} rows')
dim_product.show()

## dim_branch

In [ ]:
dim_branch = spark.table('bronze_branches') \
    .drop('_ingested_at','_source_file') \
    .dropDuplicates(['BranchID']) \
    .withColumn('_updated_at', F.current_timestamp())

dim_branch.write.format('delta').mode('overwrite').saveAsTable('dim_branch')
print(f'dim_branch: {dim_branch.count()} rows')
dim_branch.show()

## dim_date — Time Intelligence

In [ ]:
# Generate every date from 2020-01-01 to 2025-12-31
dim_date = spark.sql("SELECT explode(sequence(to_date('2020-01-01'), to_date('2025-12-31'), interval 1 day)) AS Date") \
    .withColumn('DateKey',      F.date_format('Date','yyyyMMdd').cast('int')) \
    .withColumn('Year',         F.year('Date')) \
    .withColumn('Quarter',      F.quarter('Date')) \
    .withColumn('QuarterName',  F.concat(F.lit('Q'), F.quarter('Date').cast('string'))) \
    .withColumn('Month',        F.month('Date')) \
    .withColumn('MonthName',    F.date_format('Date','MMMM')) \
    .withColumn('MonthShort',   F.date_format('Date','MMM')) \
    .withColumn('Week',         F.weekofyear('Date')) \
    .withColumn('DayOfMonth',   F.dayofmonth('Date')) \
    .withColumn('DayOfWeek',    F.dayofweek('Date')) \
    .withColumn('DayName',      F.date_format('Date','EEEE')) \
    .withColumn('IsWeekend',    F.when(F.dayofweek('Date').isin([1,7]), True).otherwise(False)) \
    .withColumn('FiscalYear',   F.when(F.month('Date') >= 10, F.year('Date') + 1).otherwise(F.year('Date'))) \
    .withColumn('FiscalQuarter',F.when(F.month('Date').isin([10,11,12]), 1)
                                 .when(F.month('Date').isin([1,2,3]), 2)
                                 .when(F.month('Date').isin([4,5,6]), 3)
                                 .otherwise(4))

dim_date.write.format('delta').mode('overwrite').saveAsTable('dim_date')
print(f'dim_date: {dim_date.count()} rows (2020-2025)')
dim_date.show(3)

## Silver Summary + Quality Checks

In [ ]:
%%sql
SELECT 'dim_customer' AS DimTable, COUNT(*) AS Rows FROM dim_customer UNION ALL
SELECT 'dim_account',               COUNT(*)         FROM dim_account  UNION ALL
SELECT 'dim_product',               COUNT(*)         FROM dim_product  UNION ALL
SELECT 'dim_branch',                COUNT(*)         FROM dim_branch   UNION ALL
SELECT 'dim_date',                  COUNT(*)         FROM dim_date
ORDER BY Rows DESC

In [ ]:
%%sql
-- Referential integrity check: accounts must have valid customers
SELECT 'Orphan Accounts' AS Check, COUNT(*) AS Count
FROM dim_account a
LEFT JOIN dim_customer c ON a.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL
UNION ALL
SELECT 'Valid Accounts', COUNT(*)
FROM dim_account a
INNER JOIN dim_customer c ON a.CustomerID = c.CustomerID